# Problem 12 (100 points)

This is a comprehensive, contest-level deep learning problem that integrates multiple concepts: custom layer design, architecture engineering under parameter constraints, custom loss functions, and model analysis. It mirrors the difficulty and format of USAAIO Round 2.

We use the following notation in this problem.
- $x \in \mathbb{R}^{B \times C \times H \times W}$ — feature maps.
- $\text{GAP}(x)$ — Global Average Pooling: $\mathbb{R}^{B \times C \times H \times W} \to \mathbb{R}^{B \times C}$.
- SE block — Squeeze-and-Excitation: channel attention via $\text{GAP} \to \text{FC} \to \text{ReLU} \to \text{FC} \to \text{Sigmoid} \to \text{scale}$.
- $r$ — SE reduction ratio (default 16).

In [ ]:
# Run code in this cell

"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)

> WARNING !!!
>
- Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else for the following purposes**:
    - **As a part of your final solution.**
    - **Temporarily import something to assist you to get a solution.**

## Part 1 (15 points, coding task)

**Do the following tasks.**

**Squeeze-and-Excitation (SE) Block.**

Implement an SE block that adaptively recalibrates channel-wise feature responses:

```
x: (B, C, H, W)
  -> GAP -> (B, C)
  -> Linear(C, C//r) -> ReLU -> Linear(C//r, C) -> Sigmoid -> (B, C)
  -> unsqueeze to (B, C, 1, 1)
  -> elementwise multiply with original x -> (B, C, H, W)
```

The two `Linear` layers should have bias.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        ...
    
    def forward(self, x):
        """x: (B, C, H, W) -> (B, C, H, W)"""
        ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
se = SEBlock(64, reduction=16)
x = torch.randn(2, 64, 8, 8)
out = se(x)
assert out.shape == (2, 64, 8, 8), f"Expected (2,64,8,8), got {out.shape}"

se_params = sum(p.numel() for p in se.parameters())
expected = 64 * (64 // 16) + (64 // 16) + (64 // 16) * 64 + 64  # W1 + b1 + W2 + b2
assert se_params == expected, f"SE params: expected {expected}, got {se_params}"
print(f"Part 1 passed! SE block params: {se_params}")

Now let us combine the SE block with residual connections to build a powerful yet compact architecture.

## Part 2 (20 points, coding task)

**Do the following tasks.**

Design an `SEResNet` for CIFAR-10 under a strict **parameter budget**.

Requirements:
- Input: $(B, 3, 32, 32)$. Output: $(B, 10)$ logits.
- Total parameters: between **100,000 and 500,000**.
- Must use at least 2 residual blocks, each containing an SE block.
- Must use Global Average Pooling before the classifier (no large FC layers).
- All convolutions with `bias=False` (use BN instead).

First implement `SEResBlock`:
```
x -> Conv(3x3) -> BN -> ReLU -> Conv(3x3) -> BN -> SE -> (+shortcut) -> ReLU -> y
```
Then assemble `SEResNet`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class SEResBlock(nn.Module):
    """Residual block with SE attention."""
    def __init__(self, in_channels, out_channels, stride=1, reduction=16):
        super().__init__()
        ...
    
    def forward(self, x):
        ...

class SEResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        ...
    
    def forward(self, x):
        """x: (B, 3, 32, 32) -> (B, 10)"""
        ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
model = SEResNet()
x = torch.randn(4, 3, 32, 32)
logits = model(x)
assert logits.shape == (4, 10), f"Expected (4, 10), got {logits.shape}"

total_params = sum(p.numel() for p in model.parameters())
assert 100_000 <= total_params <= 500_000, \
    f"Params {total_params:,} must be between 100K and 500K"

# Gradients should flow
logits.sum().backward()
assert all(p.grad is not None for p in model.parameters())

print(f"Part 2 passed! SEResNet params: {total_params:,}")

With the architecture in place, let us build a custom loss function that improves generalization.

## Part 3 (15 points, coding task)

**Do the following tasks.**

Implement **Label Smoothing Cross-Entropy Loss**.

Smoothed targets:

$$y_i^{\text{smooth}} = \begin{cases} 1 - \epsilon + \frac{\epsilon}{K} & i = \text{target} \\ \frac{\epsilon}{K} & i \neq \text{target} \end{cases}$$

Loss: $L = -\sum_{i=1}^{K} y_i^{\text{smooth}} \log p_i$ where $p = \text{softmax}(\text{logits})$.

When $\epsilon = 0$, the loss must exactly match `nn.CrossEntropyLoss`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class LabelSmoothingCE(nn.Module):
    def __init__(self, num_classes, smoothing=0.1):
        super().__init__()
        ...
    
    def forward(self, logits, targets):
        """logits: (B, K), targets: (B,) -> scalar"""
        ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
logits = torch.randn(8, 10, requires_grad=True)
targets = torch.randint(0, 10, (8,))

criterion_ls = LabelSmoothingCE(10, smoothing=0.1)
criterion_ce = nn.CrossEntropyLoss()

loss_ls = criterion_ls(logits, targets)
loss_ce = criterion_ce(logits, targets)

assert loss_ls.dim() == 0
assert loss_ls.requires_grad

# smoothing=0 must match CE
criterion_ls0 = LabelSmoothingCE(10, smoothing=0.0)
assert torch.allclose(criterion_ls0(logits, targets), loss_ce, atol=1e-5)

print(f"Part 3 passed! LS: {loss_ls.item():.4f}, CE: {loss_ce.item():.4f}")

In competition settings, you sometimes receive a black-box model and must analyze it programmatically.

## Part 4 (15 points, coding task)

**Do the following tasks.**

Implement `analyze_model(model)` that inspects any `nn.Module` and returns a dict containing:

- `'total_params'`: int — total number of parameters.
- `'has_batchnorm'`: bool — whether the model contains any `BatchNorm` layer.
- `'num_conv_layers'`: int — number of `Conv2d` layers.
- `'num_linear_layers'`: int — number of `Linear` layers.
- `'output_shape'`: tuple — output shape for input `(1, 3, 32, 32)`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def analyze_model(model):
    """
    Analyze a model's architecture.
    Returns dict with 'total_params', 'has_batchnorm', 'num_conv_layers',
                       'num_linear_layers', 'output_shape'
    """
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
test_model = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
    nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10))

info = analyze_model(test_model)
assert info['total_params'] == sum(p.numel() for p in test_model.parameters())
assert info['has_batchnorm'] == True
assert info['num_conv_layers'] == 2
assert info['num_linear_layers'] == 1
assert tuple(info['output_shape']) == (1, 10) or info['output_shape'] == torch.Size([1, 10])
print(f"Part 4 passed! {info}")

The next part tests whether you can compute parameter counts quickly by hand — a critical USAAIO Round 1 skill.

## Part 5 (10 points, non-coding task)

**Do the following tasks (Reasoning is required).**

Compute the **exact** parameter count for this architecture (input $3 \times 32 \times 32$):

```
Conv2d(3, 32, 3, pad=1, bias=False) -> BN(32) -> ReLU
-> SEResBlock(32, 32)   [2 x Conv(32,32,3,pad=1,bias=False) + 2 BN(32) + SE(32,r=16)]
-> SEResBlock(32, 64, stride=2)  [projection shortcut + SE(64,r=16)]
-> GAP -> Linear(64, 10)
```

Break down the count layer by layer. Store the total as `total_params_manual` (int).

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

In [ ]:
### WRITE YOUR SOLUTION HERE ###

total_params_manual = ...  # int

""" END OF THIS PART """

## Part 6 (10 points, non-coding task)

**Do the following tasks (Reasoning is required).**

1. The SE block applies **input-dependent** channel scaling via `sigmoid(FC(ReLU(FC(GAP(x)))))`. Why is this more powerful than a fixed learnable scale $\gamma$ per channel (as in batch normalization)?

2. Label smoothing changes the gradient of the correct class from $(p_k - 1)$ to $(p_k - 1 + \epsilon)$ and incorrect classes from $p_j$ to $(p_j - \epsilon/K)$. What is the effect on the model’s confidence? Why does this improve generalization?

3. You have a parameter budget of 1M for a CIFAR-10 CNN. Compare two designs: (a) 4 layers with 256 channels each, (b) 8 layers with 128 channels each. Both use $3 \times 3$ convolutions. Which has more parameters? Which is likely to perform better and why?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 7 (15 points, coding task)

**Do the following tasks.**

**Knowledge distillation loss.** In knowledge distillation, a small "student" model learns from both the ground-truth labels and the soft predictions of a larger "teacher" model.

The distillation loss is:

$$L = (1 - \lambda) \cdot L_{\text{CE}}(p_s, y) + \lambda \cdot T^2 \cdot \text{KL}\!\left(\text{softmax}(z_t / T) \;\|\; \text{softmax}(z_s / T)\right)$$

where $z_t, z_s$ are teacher/student logits, $T$ is the temperature, $\lambda$ is the balance factor, and KL is the KL divergence.

Implement `DistillationLoss(temperature, alpha)` as an `nn.Module`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class DistillationLoss(nn.Module):
    def __init__(self, temperature=4.0, alpha=0.7):
        """
        temperature: softmax temperature T
        alpha: weight for distillation loss (1-alpha for CE)
        """
        super().__init__()
        ...
    
    def forward(self, student_logits, teacher_logits, targets):
        """
        student_logits: (B, K)
        teacher_logits: (B, K)
        targets: (B,)
        Returns: scalar loss
        """
        ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
student_logits = torch.randn(8, 10, requires_grad=True)
teacher_logits = torch.randn(8, 10)
targets = torch.randint(0, 10, (8,))

kd_loss = DistillationLoss(temperature=4.0, alpha=0.7)
loss = kd_loss(student_logits, teacher_logits, targets)

assert loss.dim() == 0, "Loss should be scalar"
assert loss.requires_grad, "Loss should be differentiable"
assert loss.item() > 0, "Loss should be positive"

# With alpha=0, should be pure CE
kd_ce = DistillationLoss(temperature=4.0, alpha=0.0)
loss_ce_only = kd_ce(student_logits, teacher_logits, targets)
ce_ref = nn.CrossEntropyLoss()(student_logits, targets)
assert torch.allclose(loss_ce_only, ce_ref, atol=1e-5), "alpha=0 should give pure CE"

print(f"Part 7 passed! KD loss: {loss.item():.4f}")